In [1]:
def init_function_probs(conv_mass, pool_mass, noop_mass):
    # Original dictionary
    function_dict = {
        'conv_1_1_32':  {'function': 'ConvBlock', 'params': {'kernel': 1, 'strides': 1, 'filters': 32},  'prob': None},
        'conv_1_1_64':  {'function': 'ConvBlock', 'params': {'kernel': 1, 'strides': 1, 'filters': 64},  'prob': None},
        'conv_3_1_32':  {'function': 'ConvBlock', 'params': {'kernel': 3, 'strides': 1, 'filters': 32},  'prob': None},
        'conv_3_1_64':  {'function': 'ConvBlock', 'params': {'kernel': 3, 'strides': 1, 'filters': 64},  'prob': None},
        'conv_3_1_128': {'function': 'ConvBlock', 'params': {'kernel': 3, 'strides': 1, 'filters': 128}, 'prob': None},
        'conv_3_1_256': {'function': 'ConvBlock', 'params': {'kernel': 3, 'strides': 1, 'filters': 256}, 'prob': None},
        'conv_5_1_32':  {'function': 'ConvBlock', 'params': {'kernel': 5, 'strides': 1, 'filters': 32},  'prob': None},
        'conv_5_1_64':  {'function': 'ConvBlock', 'params': {'kernel': 5, 'strides': 1, 'filters': 64},  'prob': None},
        'max_pool_2_2': {'function': 'MaxPooling', 'params': {'kernel': 2, 'strides': 2}, 'prob': None},
        'avg_pool_2_2': {'function': 'AvgPooling', 'params': {'kernel': 2, 'strides': 2}, 'prob': None},
        'no_op':        {'function': 'NoOp', 'params': {}, 'prob': None}
    }

    # Split into categories
    conv_keys = [k for k, v in function_dict.items() if v["function"] == "ConvBlock"]
    pool_keys = [k for k, v in function_dict.items() if "Pooling" in v["function"]]
    noop_key = [k for k, v in function_dict.items() if v["function"] == "NoOp"][0]

    # Desired total mass allocation
    conv_mass = conv_mass
    pool_mass = pool_mass
    noop_mass = noop_mass

    # Per-function probabilities
    conv_prob = conv_mass / len(conv_keys)
    pool_prob = pool_mass / len(pool_keys)

    # Assign
    for k in conv_keys:
        function_dict[k]["prob"] = conv_prob
    for k in pool_keys:
        function_dict[k]["prob"] = pool_prob
    function_dict[noop_key]["prob"] = noop_mass

    return function_dict


In [2]:
# Desired total mass allocation
conv_mass = 0.60
pool_mass = 0.25
noop_mass = 0.15

fdict = init_function_probs(conv_mass, pool_mass, noop_mass)
print("Initialized function probabilities:\n")
for k, v in fdict.items():
    print(f"{k:15s} -> {v['prob']:.4f}")
# Sanity check: sum should be ~1.0
total_prob = sum(v["prob"] for v in fdict.values())
print(f"\nTotal probability mass = {total_prob:.4f}")


Initialized function probabilities:

conv_1_1_32     -> 0.0750
conv_1_1_64     -> 0.0750
conv_3_1_32     -> 0.0750
conv_3_1_64     -> 0.0750
conv_3_1_128    -> 0.0750
conv_3_1_256    -> 0.0750
conv_5_1_32     -> 0.0750
conv_5_1_64     -> 0.0750
max_pool_2_2    -> 0.1250
avg_pool_2_2    -> 0.1250
no_op           -> 0.1500

Total probability mass = 1.0000


In [3]:
# qnas_config_utils.py
import numpy as np
from collections import defaultdict

In [4]:
# --- Utilities ---------------------------------------------------------------

def _normalize_masses(masses: dict) -> dict:
    """Renormalize category masses to sum to 1.0 if they don't already."""
    s = float(sum(masses.values()))
    if s <= 0:
        raise ValueError("Category masses must sum to a positive value.")
    if not np.isclose(s, 1.0, atol=1e-12):
        masses = {k: v / s for k, v in masses.items()}
    return masses

def normalize_probs(function_dict: dict) -> dict:
    """Normalize all function probs to sum exactly 1.0 (handles float drift)."""
    total = float(sum(v["prob"] for v in function_dict.values()))
    if total <= 0:
        raise ValueError("Total probability <= 0 after assignment.")
    inv = 1.0 / total
    for k in function_dict:
        # clamp tiny negatives due to numerical noise, then renormalize
        function_dict[k]["prob"] = max(function_dict[k]["prob"], 0.0) * inv
    return function_dict

def check_fn_dict(function_dict: dict, tol: float = 1e-8) -> None:
    total = float(sum(v["prob"] for v in function_dict.values()))
    if not np.isclose(total, 1.0, atol=tol):
        raise ValueError(
            f"Function probabilities should sum 1.0! Got {total:.12f}. Tolerance {tol}."
        )

# --- Category mapping --------------------------------------------------------

def default_category_of(function_name: str) -> str:
    """
    Map function types to categories used for mass allocation.
    Adjust here if you add new operator families.
    """
    if function_name == "NoOp":
        return "noop"
    if "Pooling" in function_name:
        return "pool"
    if "Residual" in function_name:
        return "residual"
    if "Conv" in function_name:
        return "conv"
    return "other"  # gets zero unless you give it mass explicitly

# --- Core allocator ----------------------------------------------------------

def allocate_category_masses(function_dict: dict, category_masses: dict, category_of=default_category_of) -> dict:
    """
    Distribute desired mass per category evenly among functions in that category.
    Then normalize across all ops to ensure exact sum==1.0.
    """
    # 1) optional safety: renormalize the desired masses if they're slightly off
    category_masses = _normalize_masses(dict(category_masses))

    # 2) group keys by category
    keys_by_cat = defaultdict(list)
    for k, v in function_dict.items():
        function_dict[k]["prob"] = 0.0  # initialize
        cat = category_of(v["function"])
        keys_by_cat[cat].append(k)

    # 3) allocate evenly inside each category
    for cat, mass in category_masses.items():
        keys = keys_by_cat.get(cat, [])
        if not keys:
            continue
        per = mass / len(keys)
        for k in keys:
            function_dict[k]["prob"] = per

    # 4) normalize to kill any floating drift and pass strict checks
    function_dict = normalize_probs(function_dict)

    # 5) final guard
    check_fn_dict(function_dict, tol=1e-8)
    return function_dict

# --- Example builders (adjust as needed) ------------------------------------

def build_conv_pool_noop_dict() -> dict:
    return {
        'conv_1_1_32':  {'function': 'ConvBlock', 'params': {'kernel': 1, 'strides': 1, 'filters': 32},  'prob': None},
        'conv_1_1_64':  {'function': 'ConvBlock', 'params': {'kernel': 1, 'strides': 1, 'filters': 64},  'prob': None},
        'conv_3_1_32':  {'function': 'ConvBlock', 'params': {'kernel': 3, 'strides': 1, 'filters': 32},  'prob': None},
        'conv_3_1_64':  {'function': 'ConvBlock', 'params': {'kernel': 3, 'strides': 1, 'filters': 64},  'prob': None},
        'conv_3_1_128': {'function': 'ConvBlock', 'params': {'kernel': 3, 'strides': 1, 'filters': 128}, 'prob': None},
        'conv_3_1_256': {'function': 'ConvBlock', 'params': {'kernel': 3, 'strides': 1, 'filters': 256}, 'prob': None},
        'conv_5_1_32':  {'function': 'ConvBlock', 'params': {'kernel': 5, 'strides': 1, 'filters': 32},  'prob': None},
        'conv_5_1_64':  {'function': 'ConvBlock', 'params': {'kernel': 5, 'strides': 1, 'filters': 64},  'prob': None},
        'max_pool_2_2': {'function': 'MaxPooling', 'params': {'kernel': 2, 'strides': 2}, 'prob': None},
        'avg_pool_2_2': {'function': 'AvgPooling', 'params': {'kernel': 2, 'strides': 2}, 'prob': None},
        'no_op':        {'function': 'NoOp', 'params': {}, 'prob': None},
    }

def build_residual_pool_noop_dict() -> dict:
    return {
        # Residual V1
        'bv1_3_1_64':   {'function': 'ResidualV1', 'params': {'kernel': 3, 'strides': 1, 'filters': 64},   'prob': None},
        'bv1_3_1_128':  {'function': 'ResidualV1', 'params': {'kernel': 3, 'strides': 1, 'filters': 128},  'prob': None},
        'bv1_3_1_256':  {'function': 'ResidualV1', 'params': {'kernel': 3, 'strides': 1, 'filters': 256},  'prob': None},

        # Residual V1 + CBAM
        'resv1cbam_3_1_64':   {'function': 'ResidualV1CBAM', 'params': {'kernel': 3, 'strides': 1, 'filters': 64},  'prob': None},
        'resv1cbam_3_1_128':  {'function': 'ResidualV1CBAM', 'params': {'kernel': 3, 'strides': 1, 'filters': 128}, 'prob': None},
        'resv1cbam_3_1_256':  {'function': 'ResidualV1CBAM', 'params': {'kernel': 3, 'strides': 1, 'filters': 256}, 'prob': None},


        # Pooling + No-op
        'max_pool_2_2': {'function': 'MaxPooling', 'params': {'kernel': 2, 'strides': 2}, 'prob': None},
        'avg_pool_2_2': {'function': 'AvgPooling', 'params': {'kernel': 2, 'strides': 2}, 'prob': None},
        'no_op':        {'function': 'NoOp', 'params': {}, 'prob': None},
    }


In [5]:
# 1) conv / pool / noop like your first snippet
fdict = build_conv_pool_noop_dict()
masses = {"conv": 0.60, "pool": 0.25, "noop": 0.15}
fdict = allocate_category_masses(fdict, masses)    # will pass check_fn_dict

# 2) residual (incl. ResidualV1 + ResidualV1CBAM) / pool / noop
fdict2 = build_residual_pool_noop_dict()
masses2 = {"residual": 0.60, "pool": 0.25, "noop": 0.15}
fdict2 = allocate_category_masses(fdict2, masses2) # will pass check_fn_dict


In [6]:
fdict2

{'bv1_3_1_64': {'function': 'ResidualV1',
  'params': {'kernel': 3, 'strides': 1, 'filters': 64},
  'prob': 0.09999999999999999},
 'bv1_3_1_128': {'function': 'ResidualV1',
  'params': {'kernel': 3, 'strides': 1, 'filters': 128},
  'prob': 0.09999999999999999},
 'bv1_3_1_256': {'function': 'ResidualV1',
  'params': {'kernel': 3, 'strides': 1, 'filters': 256},
  'prob': 0.09999999999999999},
 'resv1cbam_3_1_64': {'function': 'ResidualV1CBAM',
  'params': {'kernel': 3, 'strides': 1, 'filters': 64},
  'prob': 0.09999999999999999},
 'resv1cbam_3_1_128': {'function': 'ResidualV1CBAM',
  'params': {'kernel': 3, 'strides': 1, 'filters': 128},
  'prob': 0.09999999999999999},
 'resv1cbam_3_1_256': {'function': 'ResidualV1CBAM',
  'params': {'kernel': 3, 'strides': 1, 'filters': 256},
  'prob': 0.09999999999999999},
 'max_pool_2_2': {'function': 'MaxPooling',
  'params': {'kernel': 2, 'strides': 2},
  'prob': 0.125},
 'avg_pool_2_2': {'function': 'AvgPooling',
  'params': {'kernel': 2, 'strides'